# NISAR GCOV — HDF5 Product Explorer


# 03 — L1 Metadata-First Explorer

Metadata gets read before any pixel data does — shape, units, validity, all of it — so nothing downstream gets loaded blind.

In [ ]:
from pathlib import Path
from nisar_utils.bootstrap import setup_workshop
WORKSHOP_ROOT = setup_workshop()

from nisar_utils.config import load_config
from nisar_utils.workflow import (
    build_profile, resolve_frequency, resolve_terms,
    resolve_aoi, resolve_window
)

cfg = load_config()
NISAR_FILE = Path(cfg["nisar_file"])
profile = build_profile(cfg)
freq = resolve_frequency(cfg, profile)
terms, diagonal_terms, off_diagonal_terms = resolve_terms(profile, freq)

print("File:", NISAR_FILE)
print("SAR family:", profile.sar_family)
print("Band:", profile.band)
print("Level:", profile.product_level)
print("Product:", profile.product_type)
print("GCOV root:", profile.gcov_root)
print("Frequency:", freq)
print("Polarization:", profile.polarization_channels)


## Geographic context

Everything inspected below is still anchored to the scene footprint from Module 01.

In [ ]:
from nisar_utils.gcov import open_gcov,get_grid_coordinates
from nisar_utils.mapping import scene_extent_wgs84,plot_scene_overview,folium_scene_map
grid=f"{profile.gcov_root}/grids/{freq}"
with open_gcov(NISAR_FILE) as f: _x,_y=get_grid_coordinates(f,grid)
scene_bounds,_=scene_extent_wgs84(_x,_y,profile.epsg)
print("Scene WGS84 extent:",scene_bounds)


## Optional: bring your own GIS layer

If you have a GeoJSON, Shapefile, or GeoPackage handy, you can overlay it on the footprint here — it's reprojected to WGS84 automatically. Just hit Enter if you'd rather skip this.

In [ ]:
from nisar_utils.mapping import load_vector_layer
GIS_PATH=input("Optional GIS layer path (Enter to skip): ").strip().strip('"')
if GIS_PATH:
    user_layer=load_vector_layer(GIS_PATH)
    gis_map=folium_scene_map(_x,_y,profile.epsg,title="NISAR footprint + user GIS layer",gis_layers={"User GIS layer":user_layer})
    gis_map
else:
    print("No user GIS layer selected; continuing with NISAR footprint map.")


In [ ]:
plot_scene_overview(_x,_y,profile.epsg,title=f"NISAR {profile.sar_family} {freq} — Geographic Context")


In [ ]:
m=folium_scene_map(_x,_y,profile.epsg,title="NISAR scene — geographic context")
m


In [ ]:
from nisar_utils.hdf5 import list_tree
from nisar_utils.gcov import open_gcov

with open_gcov(NISAR_FILE) as f:
    print("Root objects:", list(f.keys()))
    tree = list_tree(f, 3)
print("\nFirst tree entries:")
for name, kind in tree[:80]:
    print(f"{kind:8s} {name}")


In [ ]:
print("\nGCOV frequencies:", profile.frequencies)
for this_freq in profile.frequencies:
    print(this_freq, "terms:", profile.covariance_terms.get(this_freq, []))
print("\nModule 03 STATUS: PASS")
